In [2]:
# --- BƯỚC 0: SETUP ---
!pip uninstall -y tensorflow pyarrow
!pip install "pyarrow<20.0.0"
!pip install -U -q git+https://github.com/illuin-tech/colpali
!pip install -U -q transformers accelerate peft bitsandbytes

import torch
import warnings
import os
import gc
warnings.filterwarnings('ignore')


In [ ]:
# --- BƯỚC 1: SETUP DATA (BATCH MODE) ---
import glob
import os
import json
import io
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

# Chỉnh mốc batch tại đây
START_IDX = 280
END_IDX = 313

SEARCH_ROOT = "/kaggle/input"
ANNOTATIONS_PATH = "/kaggle/input/mmdocir-eval-data/MMDocIR_annotations.jsonl"
PARQUET_PATH = "/kaggle/input/mmdocir-eval-data/MMDocIR_layouts.parquet"
ENHANCED_JSONL_DIR = None
ENHANCED_IMG_DIR = None

for root, dirs, files in os.walk(SEARCH_ROOT):
    if "LAYOUT_CONTENT_FINAL" in root:
        ENHANCED_JSONL_DIR = root
    if "IMAGE ENHACED" in root or sum(1 for f in files if f.endswith(".jpg")) > 1000:
        ENHANCED_IMG_DIR = root

if not ENHANCED_JSONL_DIR:
    ENHANCED_JSONL_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/LAYOUT_CONTENT_FINAL"
if not ENHANCED_IMG_DIR:
    ENHANCED_IMG_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/IMAGE ENHACED"

print(f"Enhanced JSONL dir: {ENHANCED_JSONL_DIR}")
print(f"Enhanced IMG dir:   {ENHANCED_IMG_DIR}")

# A. Doc có câu hỏi trong annotations
valid_docs_in_qa = set()
with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        try:
            d = json.loads(line)
            valid_docs_in_qa.add(d["doc_name"].replace(".pdf", ""))
        except:
            pass

# B. Doc có file jsonl enrichment
available_jsonls = glob.glob(os.path.join(ENHANCED_JSONL_DIR, "*.jsonl"))
jsonl_map = {}
for p in available_jsonls:
    fname = os.path.basename(p).replace("_layout.jsonl", "")
    jsonl_map[fname] = p

intersection_docs = sorted(list(valid_docs_in_qa.intersection(jsonl_map.keys())))
if not intersection_docs:
    raise ValueError("Không tìm thấy tài liệu chung giữa annotations và enrichment JSONL")

target_doc_names = intersection_docs[START_IDX:END_IDX]
target_files = [jsonl_map[d] for d in target_doc_names]

print(f"PROCESSING BATCH [{START_IDX}:{END_IDX}]")
print(f"Selected docs ({len(target_doc_names)}): {target_doc_names}")
if len(target_doc_names) == 0:
    raise ValueError("Batch rỗng, kiểm tra START_IDX/END_IDX")

print("Loading backbone parquet...")
df_orig = pd.read_parquet(PARQUET_PATH)
df_orig["join_doc_name"] = df_orig["doc_name"].str.replace(".pdf", "", regex=False)
df_orig = df_orig[df_orig["join_doc_name"].isin(target_doc_names)]

print("Loading enrichment jsonl...")
dfs = []
for f in tqdm(target_files, desc="Reading JSONLs"):
    try:
        temp = pd.read_json(f, lines=True)
        temp["join_doc_name"] = os.path.basename(f).replace("_layout.jsonl", "")
        if "layout" in temp.columns:
            temp = temp.rename(columns={"layout": "layout_id"})

        cols = ["join_doc_name", "layout_id", "vlm_text", "img_enhanced_path"]
        if "text_level" in temp.columns:
            cols.append("text_level")
        temp = temp[[c for c in cols if c in temp.columns]]
        dfs.append(temp)
    except:
        pass

if dfs:
    df_enh = pd.concat(dfs, ignore_index=True)
    df_enh = df_enh.rename(columns={"vlm_text": "vlm_text_enhanced", "text_level": "text_level_enhanced"})
else:
    df_enh = pd.DataFrame()

print("Merging and contextualizing...")
df_final = pd.merge(df_orig, df_enh, on=["join_doc_name", "layout_id"], how="left")
df_final = df_final.sort_values(by=["join_doc_name", "page_id", "layout_id"])

def identify_header(row):
    if row.get("type") in ["title", "section_header", "header"]:
        return str(row.get("text", ""))
    if pd.notna(row.get("text_level_enhanced")):
        return str(row.get("text", ""))
    return np.nan

df_final["temp_header"] = df_final.apply(identify_header, axis=1)
df_final["current_section"] = df_final.groupby("join_doc_name")["temp_header"].ffill().fillna("General Content")

enh_image_map = {}
if os.path.exists(ENHANCED_IMG_DIR):
    for f in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
        enh_image_map[os.path.basename(f)] = f

def get_best_sources(row):
    img_type, img_data = None, None
    if pd.notna(row.get("img_enhanced_path")):
        fname = os.path.basename(str(row["img_enhanced_path"]))
        if fname in enh_image_map:
            img_type, img_data = "path", enh_image_map[fname]
    if img_data is None and pd.notna(row.get("image_binary")):
        img_type, img_data = "binary", row["image_binary"]

    raw_content = ""
    if pd.notna(row.get("vlm_text_enhanced")) and len(str(row["vlm_text_enhanced"])) > 5:
        raw_content = str(row["vlm_text_enhanced"])
    elif pd.notna(row.get("text")) and len(str(row["text"])) > 5:
        raw_content = str(row["text"])
    elif pd.notna(row.get("ocr_text")) and len(str(row["ocr_text"])) > 5:
        raw_content = str(row["ocr_text"])
    elif pd.notna(row.get("vlm_text")):
        raw_content = str(row["vlm_text"])

    section = row.get("current_section", "")
    final_text_prompt = f"Section: {section}\nContent: {raw_content}"
    if len(final_text_prompt) < 10:
        final_text_prompt = "Document layout."

    return pd.Series([img_type, img_data, final_text_prompt], index=["img_type", "img_data", "final_text"] )

processed = df_final.apply(get_best_sources, axis=1)
sample_layouts_df = pd.concat([df_final, processed], axis=1).dropna(subset=["img_type"]).reset_index(drop=True)

print("=" * 60)
print(f"BATCH READY: [{START_IDX}:{END_IDX}]")
print(f"Docs: {target_doc_names}")
print(f"Total layouts: {len(sample_layouts_df)}")
print("=" * 60)

In [ ]:
# --- BƯỚC 2: LOAD MODEL ---
print(">>> BƯỚC 2: Loading ColSmolVLM-500M...")

import torch
import gc
import os
from colpali_engine.models import ColIdefics3, ColIdefics3Processor

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "vidore/colSmol-500M"

model = ColIdefics3.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="eager" 
).eval()

processor = ColIdefics3Processor.from_pretrained(MODEL_NAME)

print(" Model & Processor Ready!")

In [ ]:
# --- BƯỚC 3: FUSED INDEXING ---
print(">>> BƯỚC 3: Creating Fused Vectors")

import io
import os
import gc
import pickle
import torch
from PIL import Image
from tqdm.notebook import tqdm

WORKING_DIR = "/kaggle/working"
INDEX_SUFFIX = f"{START_IDX}_{END_IDX}" if "START_IDX" in globals() and "END_IDX" in globals() else "full"
FINAL_INDEX_PATH = os.path.join(WORKING_DIR, f"colsmol_fused_index_{INDEX_SUFFIX}.pkl")
CHECKPOINT_PATH = os.path.join(WORKING_DIR, f"colsmol_checkpoint_{INDEX_SUFFIX}.pkl")
BATCH_SIZE = 4
SAVE_EVERY = 500

fused_index = []
start_idx = 0
skip_encoding = False

if os.path.exists(FINAL_INDEX_PATH):
    print(f" Found completed index: {FINAL_INDEX_PATH}")
    with open(FINAL_INDEX_PATH, "rb") as f:
        loaded = pickle.load(f)
    fused_index = loaded["fused_index"] if isinstance(loaded, dict) and "fused_index" in loaded else loaded
    print(f" Loaded {len(fused_index)} layouts. Skip encoding.")
    skip_encoding = True
elif os.path.exists(CHECKPOINT_PATH):
    print(" Found checkpoint. Resuming...")
    try:
        with open(CHECKPOINT_PATH, "rb") as f:
            ckpt = pickle.load(f)
        if isinstance(ckpt, dict):
            fused_index = ckpt.get("fused_index", [])
            start_idx = int(ckpt.get("next_row", len(fused_index)))
        else:
            fused_index = ckpt
            start_idx = len(fused_index)
        print(f" Restored {len(fused_index)} layouts. Continue from row {start_idx}.")
    except Exception as e:
        print(f" Checkpoint corrupted ({e}). Start from scratch.")
        fused_index, start_idx = [], 0
else:
    print(" Fresh run.")

if not skip_encoding:
    if "sample_layouts_df" not in globals():
        raise ValueError("sample_layouts_df not found. Run BƯỚC 1 first.")

    print(f" Total layouts to process: {len(sample_layouts_df)}")
    remaining_df = sample_layouts_df.iloc[start_idx:].reset_index(drop=True)

    def load_img_safe(row):
        try:
            img = None
            if row["img_type"] == "path":
                img = Image.open(row["img_data"])
            elif row["img_type"] == "binary":
                img = Image.open(io.BytesIO(row["img_data"]))

            if img is not None:
                img = img.convert("RGB")
                if img.width < 14 or img.height < 14:
                    return Image.new("RGB", (224, 224), "white")
                return img
        except:
            pass
        return Image.new("RGB", (224, 224), "white")

    batch_imgs, batch_texts = [], []

    for i, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="Encoding"):
        current_real_idx = start_idx + i

        img = load_img_safe(row)
        txt = str(row["final_text"]).replace("<image>", " ")[:2000]

        batch_imgs.append(img)
        batch_texts.append(txt)

        if len(batch_imgs) >= BATCH_SIZE or i == len(remaining_df) - 1:
            with torch.no_grad():
                try:
                    inputs_vis = processor.process_images(batch_imgs).to(device)
                    out_vis = model(**inputs_vis)
                    emb_vis_list = list(torch.unbind(out_vis.to("cpu")))

                    inputs_txt = processor.process_queries(batch_texts, max_length=512, suffix="").to(device)
                    out_txt = model(**inputs_txt)
                    emb_txt_list = list(torch.unbind(out_txt.to("cpu")))

                    for k in range(len(emb_vis_list)):
                        fused_vec = torch.cat([emb_vis_list[k], emb_txt_list[k]], dim=0).to(torch.float16).contiguous()
                        fused_index.append(fused_vec)
                except Exception as e:
                    print(f" Batch {current_real_idx} error: {e}. Skip batch.")

            batch_imgs, batch_texts = [], []

            if current_real_idx % 100 == 0:
                torch.cuda.empty_cache()
                gc.collect()

            if len(fused_index) > 0 and len(fused_index) % SAVE_EVERY == 0:
                with open(CHECKPOINT_PATH, "wb") as f:
                    pickle.dump({"fused_index": fused_index, "next_row": current_real_idx + 1}, f)
                print(f" Checkpoint saved: {len(fused_index)} layouts")

    with open(FINAL_INDEX_PATH, "wb") as f:
        pickle.dump(fused_index, f)

    if os.path.exists(CHECKPOINT_PATH):
        os.remove(CHECKPOINT_PATH)

print("=" * 60)
print(" BƯỚC 3 HOÀN TẤT")
print(f" Total docs in index: {len(fused_index)}")
print(f" Index path: {FINAL_INDEX_PATH}")
print("=" * 60)

In [ ]:
# --- BƯỚC 3.5: TRIAL SCORING MODULE (TRAINED HEAD) ---
print(">>> BƯỚC 3.5: Defining TrialHead...")

import torch
import torch.nn as nn
import torch.nn.functional as F

class TrialHead(nn.Module):
    def __init__(self, embed_dim, lambda_rel=0.5):
        super().__init__()
        self.lambda_rel = lambda_rel
        self.embed_dim = embed_dim
        self.gate_mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Mish(),
            nn.Linear(embed_dim, 1),
        )
        self.relation_mlp = nn.Sequential(
            nn.Linear(2 * embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, q_embs, doc_embs_list):
        B, Nq, D = q_embs.shape
        N = len(doc_embs_list)
        device = q_embs.device

        gate_logits = self.gate_mlp(q_embs).squeeze(-1)
        q_norms = torch.norm(q_embs.detach(), dim=-1)
        valid_mask = q_norms > 1e-6

        gate_pos = F.softplus(gate_logits) + 1e-6
        gate_pos = gate_pos.masked_fill(~valid_mask, 0.0)
        w_q = gate_pos / gate_pos.sum(dim=-1, keepdim=True).clamp(min=1e-8)

        has_bigrams = Nq > 1
        if has_bigrams:
            q_bigram = torch.cat([q_embs[:, 1:], q_embs[:, :-1]], dim=-1)
            q_rel = self.relation_mlp(q_bigram)

        scores = torch.zeros(B, N, device=device)
        for d_idx in range(N):
            d_emb = doc_embs_list[d_idx]
            sim_mat = torch.matmul(q_embs, d_emb.T)
            max_sim, max_j = sim_mat.max(dim=-1)

            rel_contrib = torch.zeros(B, Nq, device=device)
            if has_bigrams:
                d_c = d_emb[max_j[:, 1:].reshape(-1)].reshape(B, Nq - 1, D)
                d_p = d_emb[max_j[:, :-1].reshape(-1)].reshape(B, Nq - 1, D)
                d_rel = self.relation_mlp(torch.cat([d_c, d_p], dim=-1))
                rel_contrib[:, 1:] = (q_rel * d_rel).sum(dim=-1)

            token_scores = max_sim + self.lambda_rel * rel_contrib
            scores[:, d_idx] = (w_q * token_scores).sum(dim=-1)

        return scores, w_q

print("✅ TrialHead ready")

In [ ]:
# --- BƯỚC 4: EVALUATION & REPORTING (TRIAL Fixed Lambda) ---
print(">>> BƯỚC 4: TRIAL Evaluation & Exporting Reports...")

import json
import os
import pickle
import torch
import pandas as pd
from tqdm.notebook import tqdm

# ============================================================================== 
# 1. SETUP
# ============================================================================== 
WORKING_DIR = "/kaggle/working"
INDEX_SUFFIX = f"{START_IDX}_{END_IDX}" if "START_IDX" in globals() and "END_IDX" in globals() else "full"
INDEX_PATH = os.path.join(WORKING_DIR, f"colsmol_fused_index_{INDEX_SUFFIX}.pkl")
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
TRIAL_WEIGHTS_PATH = "/kaggle/input/datasets/namthi/train-weight/trial_head_weights.pt"

LAMBDA_REL = 0.5
DOC_CHUNK_SIZE = 128

# Load Index
if "fused_index" not in globals() or not fused_index:
    if os.path.exists(INDEX_PATH):
        with open(INDEX_PATH, "rb") as f:
            loaded = pickle.load(f)
        fused_index = loaded["fused_index"] if isinstance(loaded, dict) and "fused_index" in loaded else loaded
    else:
        raise FileNotFoundError(f"Index file not found: {INDEX_PATH}")

# Check DataFrame
if "sample_layouts_df" not in globals():
    raise ValueError("sample_layouts_df not found. Run BƯỚC 1 first.")

if len(fused_index) == 0:
    raise ValueError("fused_index is empty.")

# Load TrialHead weights
D = fused_index[0].shape[-1]
trial_head = TrialHead(embed_dim=D, lambda_rel=LAMBDA_REL).to(device)
trial_head.load_state_dict(torch.load(TRIAL_WEIGHTS_PATH, map_location=device, weights_only=True))
trial_head.eval()
print(f" TrialHead loaded from: {TRIAL_WEIGHTS_PATH}")

# ============================================================================== 
# 2. GENERATE QA PAIRS
# ============================================================================== 
print(" Generating QA Pairs...")

def calculate_iou(box1, box2):
    b1 = list(box1) if not isinstance(box1, list) else box1
    b2 = list(box2) if not isinstance(box2, list) else box2

    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = ((b1[2] - b1[0]) * (b1[3] - b1[1])) + ((b2[2] - b2[0]) * (b2[3] - b2[1])) - inter
    return inter / union if union > 0 else 0

qa_pairs = []
doc_lookup = {k: v for k, v in sample_layouts_df.groupby("join_doc_name")}
available_docs = set(doc_lookup.keys())
print(f" Docs in RAM: {list(available_docs)}")

matched_docs = 0
with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        try:
            doc_data = json.loads(line)
        except Exception:
            continue

        target_doc = doc_data["doc_name"].replace(".pdf", "")
        if target_doc not in available_docs:
            continue

        matched_docs += 1
        doc_layouts = doc_lookup[target_doc]
        domain = doc_data.get("domain", "General")
        col_page = "page_idx" if "page_idx" in doc_layouts.columns else "page_id"

        current_doc_df = doc_layouts.copy()
        current_doc_df["safe_page"] = pd.to_numeric(current_doc_df[col_page], errors="coerce").fillna(-999).astype(int)

        for q_item in doc_data["questions"]:
            gt_indices = []
            if "layout_mapping" in q_item:
                for target in q_item["layout_mapping"]:
                    t_page, t_bbox = target["page"], target["bbox"]
                    try:
                        t_page_int = int(t_page)
                        cands1 = current_doc_df[current_doc_df["safe_page"] == t_page_int]
                        cands2 = current_doc_df[current_doc_df["safe_page"] == t_page_int - 1]
                        cands = pd.concat([cands1, cands2])

                        for idx, row in cands.iterrows():
                            if calculate_iou(row["bbox"], t_bbox) > 0.5:
                                gt_indices.append(int(idx))
                    except Exception:
                        continue

            if gt_indices:
                valid_gt = [g for g in set(gt_indices) if g < len(fused_index)]
                if valid_gt:
                    qa_pairs.append({
                        "question": q_item["Q"],
                        "gt_indices": valid_gt,
                        "doc_name": target_doc,
                        "domain": domain,
                    })

print(f" Đã quét {matched_docs} tài liệu khớp tên.")
print(f" Found {len(qa_pairs)} QA pairs valid for testing.")

# ============================================================================== 
# 3. SCORING & EXPORT
# ============================================================================== 
if len(qa_pairs) > 0:
    print("\n" + "=" * 60)
    print(f" TRIAL SCORING  lambda = {LAMBDA_REL}")
    print(f" Score(Q,D) = Base_MaxSim + {LAMBDA_REL} x Relation")
    print("=" * 60)

    all_docs = fused_index
    num_docs = len(all_docs)

    def score_in_doc_chunks(q_embs, docs, chunk_size=128):
        parts = []
        for s in range(0, len(docs), chunk_size):
            e = min(s + chunk_size, len(docs))
            doc_chunk = [d.to(device=device, dtype=torch.float32, non_blocking=True) for d in docs[s:e]]
            with torch.no_grad():
                chunk_scores, _ = trial_head(q_embs, doc_chunk)
            parts.append(chunk_scores.detach().cpu())
            del doc_chunk, chunk_scores
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        return torch.cat(parts, dim=1) if len(parts) > 1 else parts[0]

    correct_1, correct_5, correct_10 = 0, 0, 0
    total = len(qa_pairs)
    report_data = []

    BATCH_Q = 2
    pbar = tqdm(range(0, total, BATCH_Q), desc="Evaluating")

    for i in pbar:
        batch_qa = qa_pairs[i:i + BATCH_Q]
        queries = [q["question"] for q in batch_qa]

        with torch.no_grad():
            q_inputs = processor.process_queries(queries).to(device)
            q_embeddings = model(**q_inputs).float()
            scores_cpu = score_in_doc_chunks(q_embeddings, all_docs, chunk_size=DOC_CHUNK_SIZE)

        for j in range(scores_cpu.shape[0]):
            score_row = scores_cpu[j]
            k_top = 10 if num_docs >= 10 else num_docs
            top_k_indices = torch.topk(score_row, k=k_top).indices.tolist()
            gt_set = set(batch_qa[j]["gt_indices"])

            first_hit = -1
            hits_found = []
            for r, idx in enumerate(top_k_indices):
                if idx in gt_set:
                    if first_hit == -1:
                        first_hit = r + 1
                    hits_found.append(idx)

            if first_hit != -1:
                if first_hit <= 1:
                    correct_1 += 1
                if first_hit <= 5:
                    correct_5 += 1
                if first_hit <= 10:
                    correct_10 += 1

            recall_score = len(hits_found) / len(gt_set) if len(gt_set) > 0 else 0.0
            report_data.append({
                "lambda_rel": LAMBDA_REL,
                "doc_name": batch_qa[j]["doc_name"],
                "domain": batch_qa[j]["domain"],
                "question": batch_qa[j]["question"],
                "layout_retrieved": str(top_k_indices),
                "gt": str(list(gt_set)),
                "first_hit_rank": first_hit if first_hit != -1 else "Not Found",
                "recall": round(recall_score, 2),
                "is_perfect": recall_score == 1.0,
            })

        if (i + BATCH_Q) % 20 == 0:
            pbar.set_postfix({"R@10": f"{(correct_10 / min(i + BATCH_Q, total)) * 100:.1f}%"})

    print("\n" + "=" * 50)
    print(f" FINAL RESULTS  (lambda = {LAMBDA_REL})")
    print("=" * 50)
    print(f" Recall@1:  {(correct_1 / total) * 100:.2f}%")
    print(f" Recall@5:  {(correct_5 / total) * 100:.2f}%")
    print(f" Recall@10: {(correct_10 / total) * 100:.2f}%")
    print("=" * 50)

    df_report = pd.DataFrame(report_data)
    report_path = os.path.join(WORKING_DIR, f"trial_report_best_lambda_{LAMBDA_REL}.csv")
    df_report.drop(columns=["is_perfect"]).to_csv(report_path, index=False, encoding="utf-8-sig")
    print(f" Report saved: {report_path}")

    err_path = os.path.join(WORKING_DIR, f"trial_error_analysis_best_lambda_{LAMBDA_REL}.csv")
    df_report[~df_report["is_perfect"]].drop(columns=["is_perfect"]).to_csv(err_path, index=False)
    print(f" Error analysis saved: {err_path}")

    # ========================================================================== 
    # 4. DIAGNOSTIC: TRAINED TOKEN WEIGHTS FROM TRIALHEAD (w_q)
    # ========================================================================== 
    tokenizer = getattr(processor, "tokenizer", None)
    if tokenizer is None and hasattr(processor, "query_tokenizer"):
        tokenizer = getattr(processor, "query_tokenizer")

    special_ids = set()
    if tokenizer is not None:
        if hasattr(tokenizer, "all_special_ids"):
            try:
                special_ids.update(getattr(tokenizer, "all_special_ids"))
            except Exception:
                pass
        for tid in [
            getattr(tokenizer, "pad_token_id", None),
            getattr(tokenizer, "bos_token_id", None),
            getattr(tokenizer, "eos_token_id", None),
            getattr(tokenizer, "sep_token_id", None),
            getattr(tokenizer, "cls_token_id", None),
            getattr(tokenizer, "unk_token_id", None),
        ]:
            if tid is not None:
                special_ids.add(int(tid))

    def _merge_subword_weights(tokens, weights):
        merged = []
        current_tok = ""
        current_w = 0.0

        for tok, w in zip(tokens, weights):
            if tok is None:
                continue
            if tok.startswith("<") and tok.endswith(">"):
                continue

            if tok.startswith("▁") or tok.startswith("Ġ"):
                if current_tok != "":
                    merged.append((current_tok, current_w))
                current_tok = tok[1:]
                current_w = float(w)
            else:
                if current_tok == "":
                    current_tok = tok
                    current_w = float(w)
                else:
                    current_tok += tok
                    current_w += float(w)

        if current_tok != "":
            merged.append((current_tok, current_w))

        cleaned = []
        for tok, w in merged:
            t = tok.strip()
            if t == "" or t in [",", ".", ":", ";", "-", "_", "(", ")", "[", "]", "{", "}", "'", '"', "/", "?"]:
                continue
            cleaned.append((t, w))

        s = sum(w for _, w in cleaned)
        if s > 1e-12:
            cleaned = [(t, w / s) for t, w in cleaned]
        return cleaned

    def _compute_wq_only(q_embs):
        gate_logits = trial_head.gate_mlp(q_embs).squeeze(-1)
        q_norms = torch.norm(q_embs.detach(), dim=-1)
        valid_mask = q_norms > 1e-6
        gate_pos = F.softplus(gate_logits) + 1e-6
        gate_pos = gate_pos.masked_fill(~valid_mask, 0.0)
        w_q = gate_pos / gate_pos.sum(dim=-1, keepdim=True).clamp(min=1e-8)
        return w_q

    N_DIAG = min(5, len(qa_pairs))
    diag_queries = [qa_pairs[i]["question"] for i in range(N_DIAG)]

    print("\n" + "=" * 60)
    print(f" DIAGNOSTIC: Trained Token Weights w_q from TrialHead ({N_DIAG} queries)")
    print("  w_q = normalized gate output learned by 2-MLP TrialHead")
    print("=" * 60)

    with torch.no_grad():
        diag_inputs = processor.process_queries(diag_queries).to(device)
        diag_embs = model(**diag_inputs).float()
        diag_wq = _compute_wq_only(diag_embs)

    diag_ids = diag_inputs["input_ids"].detach().cpu() if "input_ids" in diag_inputs else None
    diag_wq = diag_wq.detach().cpu()

    for i, query in enumerate(diag_queries):
        if tokenizer is not None and diag_ids is not None:
            ids_i = diag_ids[i].tolist()
            toks = tokenizer.convert_ids_to_tokens(ids_i)
            ws = diag_wq[i].tolist()

            filtered_toks = []
            filtered_ws = []
            for tid, tok, w in zip(ids_i, toks, ws):
                if tid in special_ids:
                    continue
                filtered_toks.append(tok)
                filtered_ws.append(float(w))
        else:
            filtered_toks = [f"tok_{k}" for k in range(len(diag_wq[i]))]
            filtered_ws = [float(x) for x in diag_wq[i].tolist()]

        merged = _merge_subword_weights(filtered_toks, filtered_ws)
        merged.sort(key=lambda x: x[1], reverse=True)

        print(f"\n[Q{i + 1}] {query}")
        print("  " + "-" * 56)
        print(f"  {'Token':<28} {'w_q':>9}   Importance")
        print("  " + "-" * 56)
        for tok, w in merged[:15]:
            bar = "#" * min(int(w * 800), 30)
            print(f"  {tok:<28} {w:>9.5f}   {bar}")

    print("\n" + "=" * 60)
    print(" Note: This is the trained TrialHead weight, not centroid heuristic.")
    print("=" * 60)
else:
    print(" Critical Error: Zero QA Pairs found.")